# Mathematical Foundations: MLPs, RNNs, and ODE Connections

---

## 1. Standard MLP Forward Pass

A feedforward neural network (Multi-Layer Perceptron, MLP) is a composition of affine maps and nonlinearities:

$$
h^{(0)} = x, \quad 
h^{(\ell)} = \sigma\!\left(W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}\right), \quad \ell = 1, \dots, L
$$

Final output:

$$
y = h^{(L)}
$$

where:   
- $x \in \mathbb{R}^d$: input vector  
- $W^{(\ell)}, b^{(\ell)}$: learnable weights and biases  
- $\sigma(\cdot)$: nonlinear activation function (e.g. ReLU, tanh).


---

## 2. Recurrent Neural Network (RNN) Forward Pass

A vanilla RNN processes sequential input $t = 1,\dots,T$::

$$
h_t = \sigma\!\left(W_{xh} x_t + W_{hh} h_{t-1} + b_h\right)
$$

$$
y_t = W_{hy} h_t + b_y
$$

where:  
- $h_t$: hidden state at time $t$  
- $W_{xh}, W_{hh}, W_{hy}$: input, hidden, and output weight matrices  
- $b_h, b_y$: biases.

---

## 3. Euler’s Method (Numerical Integration)

For a continuous-time dynamical system:

$$
\frac{dz}{dt} = f(z(t), t), \quad z(0) = z_0
$$

Euler’s forward step with step size $(\Delta t)$:

$$
z_{t+1} = z_t + \Delta t \, f(z_t, t)
$$

---

## 4. ODE Definition and Neural Networks

The The definition of a RNN forward pass in section 2 can be written in the following residual form:

$$
h_{t+1} = h_t + f(h_t, x_{t+1}, \theta)
$$

## Euler/Residual View Holds for Any MLP

**Setup.** Consider a feed-forward MLP with layers
$$
h^{(0)} = x \in \mathbb{R}^{d_0},\qquad
h_{(\ell+1)} = \sigma_\ell\!\big(W_\ell h_{(\ell)} + b_\ell\big)\in\mathbb{R}^{d_{\ell+1}},
\quad \ell=0,\dots,L-1.
$$
- Layer widths: $(d_{(\ell)} \in \mathbb{R})$
    - the layer $(\ell)$ takes a vector in $(\mathbb{R}^{d_l})$ to one in  $(\mathbb{R}^{d_{l+1}}).$
- Layer state ("hidden" at layer $\ell$): $h_l \in \mathbb{R}^{d_l}.$
- Weights and biasses at layer $\ell: W_\ell \in \mathbb{R}^{d_{\ell + 1} \times d_l}, b_\ell \in \mathbb{R}^{d_{\ell+1}}.$
- Nonlinearlity at layer $\ell: \sigma_\ell : \mathbb{R}^{d_{l+q}} \to \mathbb{R}^{d_{l+q}}$

### Case A: Constant width (all $d_\ell = d$)
Define
$$
f_\ell(h) := \sigma_\ell(W_\ell h + b_\ell) - h.
$$
Then each layer satisfies the **Euler/residual form** with step $\Delta t=1$:
$$
h^{(\ell+1)} = h^{(\ell)} + f_\ell\!\big(h^{(\ell)}\big).
$$
This is exactly an explicit-Euler update on the state $h$.

### Case B: Varying widths ($d_\ell$ not all equal)
### The Dimensionality Issue
In many MLP $d_\ell$ can change with $\ell$ (e.g., $784 \to 512 \to 128 \to 64$)
Euler updates have the form "new = old + increment" in a single fixed space. So we first make all states live in one space. 

Define the ambient dimension:
$m$ := $max ${$d_0, d_1, .., d_L$}

### Embed and Project

For each $\ell$ define:
- Embedding
    - $E_\ell: \mathbb{R}^{d_\ell} \to \mathbb{R}^m,$    $E_\ell(h) = [\begin{matrix}h \\0_{m-d_\ell}\end{matrix}]$

    This zero-pads any $d_\ell$-vector up to length m
- Projection
    - $P_\ell: \mathbb{R}^m \to \mathbb{R}^{d_\ell}$,   $P_\ell(z) =$ the first $d_\ell$ coordinates of $z$

    This truncates a length-m vector back to the first $d_\ell$ entries

These are left/right inverses on the image of $E_\ell: P_\ell(e_\ell(h)) = h$

## Lift the Network States

Define the lifted state in the ambient space:
$z^{(\ell)} := E_\ell(h^{(\ell)}) \in \mathbb{R}^m.$

The whole forward pass acts as updates on $z^{(\ell)}$

## One layer in the Lifted Space

Start from $z^{(\ell)}$. To apply actual layer $\ell$:

1. Project to the layers input size: $P_\ell z^{(\ell)}$

2. Affine + nonlinearity: $\sigma(W_\ell(P_\ell{z^{(\ell)}}) + b_\ell) \in \mathbb{R}^{d_{\ell+1}}$

3. Embed the results to ambient size:

$\hat{z}^{(\ell+1)} := E_{\ell+1}(\sigma_\ell(W_\ell P_\ell z^{(\ell)}+b_\ell)) \in \mathbb{R}^m$

By construction, **projecting back** recovers the original layer update:

$P_{\ell+1}\hat{z}^{(\ell+1)} = \sigma_\ell(W_\ell h^{(\ell)} + b_\ell) = h^{(\ell+1)}$


## Turn It Into a Euler/Residual Step

Define the **increment field** (the "change" that the layer applies in the ambient space):
$F_\ell(z) := E_{(\ell+1)}(\sigma(W_\ell P_\ell z+b_\ell)) - z.$

Then the **Euler/residual recursion** holds:
$Z^{(\ell+1)} = z^{(\ell)} + F_\ell(z^{(\ell)})$

and projecting gives back the true MLP state:
$P_{\ell+1}z^{(\ell+1)} = h^{(\ell+1)}$

So, despite **changing widths**, every layer is "old state plus an increment" in $\mathbb{R}^m.$

## Per-Layer Step Sizes

If we want an explicit "time step" per layer (to mirror numerical ODEs), pick any $\Delta t_\ell > 0$ and write:

$z^{(\ell+1)} = z^{(\ell)} + \Delta t_\ell G_\ell(z^{(\ell)},      G_\ell := \frac{F_\ell}{\Delta t_\ell}$

This corresponds to an Eurler discretization of a continuous-depth system $\dot{z} = G(z,t)$